# EP11 — AMM Mechanics & Price Impact
**Quantifaya · DeFi Mechanics Series · Episode 11**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Godwin-88/quantifire-web/blob/main/public/notebooks/ep11-uniswap-amm.ipynb)

> **Learning objective:** Derive the constant-product AMM formula from first principles, simulate Uniswap V2 swaps, quantify price impact and impermanent loss, and compare V3 capital efficiency.

**Companion post:** [quantifaya.com/blog/ep11-how-uniswap-works-xy-k-formula](https://quantifaya.com/blog/ep11-how-uniswap-works-xy-k-formula)

---
*Quantifaya research notebooks are provided for educational purposes only. Nothing here constitutes financial or investment advice.*

## Learning Objectives

By the end of this notebook you will be able to:

1. **Derive** the constant-product invariant $x \cdot y = k$ from first principles and understand why it produces a hyperbolic price curve.
2. **Implement** a fully functional Uniswap V2 pool in Python, including swap execution, fee accounting, and reserve tracking.
3. **Quantify** price impact as a function of trade size relative to pool depth, and identify the 1% rule of thumb.
4. **Calculate** impermanent loss for any price ratio and determine the fee volume required to break even.
5. **Understand** Uniswap V3 concentrated liquidity, virtual reserves, and capital efficiency gains versus V2.
6. **Interpret** real on-chain dynamics using historical ETH, BTC, and SOL price data from Yahoo Finance.

---
**Prerequisites:** Python basics, high-school algebra, optional: introductory calculus for the derivative derivations.

## Mathematical Foundations

### The AMM Invariant

A Uniswap V2 pool holds two tokens with reserves $x$ and $y$. Every trade must satisfy:

$$x \cdot y = k \quad \text{(constant-product invariant)}$$

This constrains the pool to move along a rectangular hyperbola in $(x, y)$ space.

### Spot Price

The marginal (spot) price of token $x$ denominated in token $y$ is the negative slope of the tangent to the hyperbola:

$$P_{\text{spot}} = -\frac{dy}{dx} = \frac{y}{x} \quad \left[\frac{\text{token}_y}{\text{token}_x}\right]$$

### Swap Output (with fee $f$)

When a trader sells $\Delta x$ of token $x$, the pool charges fee $f$ on the input. The post-fee input is $\Delta x \cdot (1-f)$. Enforcing the invariant:

$$(x + \Delta x(1-f))\cdot(y - \Delta y) = k$$

Solving for the output $\Delta y$:

$$\boxed{\Delta y = \frac{y \cdot \Delta x \cdot (1-f)}{x + \Delta x \cdot (1-f)}}$$

### Price Impact

The effective price paid is $P_{\text{eff}} = \Delta y / \Delta x$. Price impact measures the slippage relative to spot:

$$\text{PI} = \frac{P_{\text{spot}} - P_{\text{eff}}}{P_{\text{spot}}} = \frac{\Delta x}{x + \Delta x} \quad (\text{ignoring fee for clarity})$$

### Impermanent Loss

Let $r = P_{\text{new}} / P_{\text{old}}$ be the price ratio. An LP's value relative to holding is:

$$\text{IL}(r) = \frac{2\sqrt{r}}{1+r} - 1$$

Key values: $r=2 \Rightarrow \text{IL} \approx -5.7\%$, $\quad r=5 \Rightarrow \text{IL} \approx -25.3\%$.

### Uniswap V3 Capital Efficiency

V3 concentrates liquidity in a price range $[P_a, P_b]$. Capital efficiency relative to V2 full-range:

$$E = \frac{1}{1 - \sqrt{P_a / P_b}}$$

For a $\pm 10\%$ range: $P_a = 0.9P$, $P_b = 1.1P$, so $E \approx 10.5\times$.

### V3 Virtual Reserves

$$x_{\text{virtual}} = x_{\text{real}} + \frac{L}{\sqrt{P_b}}, \qquad y_{\text{virtual}} = y_{\text{real}} + L\sqrt{P_a}$$

where $L = \sqrt{k}$ is the liquidity parameter.

### Arbitrage Condition

An arbitrageur profits when the AMM price diverges from the market price $P_{\text{mkt}}$:

$$\text{Profit} \approx |P_{\text{AMM}} - P_{\text{mkt}}| \cdot \Delta x - \text{gas cost}$$

Arbitrage drives $P_{\text{AMM}} \to P_{\text{mkt}}$, keeping the pool price aligned with the broader market.

In [ ]:
!pip install yfinance pandas numpy plotly --quiet

---
## ▶ Run this cell — STEP 1: Configure Your Pool

Edit `CONFIG` to change the token pair, pool depth, fee tier, or date range. Everything downstream adapts automatically.

In [ ]:
# ─────────────────────────────────────────────
# GLOBAL CONFIGURATION — edit here to customise
# ─────────────────────────────────────────────
CONFIG: dict = {
    # Pool token pair (used for labelling — real data pulled from yfinance)
    'token_x': 'ETH',
    'token_y': 'USDC',
    'fee_tier': 0.003,   # 0.01%=0.0001 | 0.05%=0.0005 | 0.30%=0.003 | 1.00%=0.01

    # Initial pool liquidity (in token units)
    'initial_x_reserve': 1000.0,       # ETH in pool
    'initial_y_reserve': 2_000_000.0,  # USDC in pool → implied price = 2000 USDC/ETH

    # Swap simulation parameters
    'trade_sizes_pct': [0.001, 0.005, 0.01, 0.05, 0.10, 0.20, 0.30],

    # V3 range (as % around current price)
    'v3_range_pct': 0.10,   # ±10% range

    # Market data
    'tickers': ['ETH-USD', 'BTC-USD', 'SOL-USD'],
    'start_date': '2022-01-01',
    'end_date': '2024-12-31',

    # Plotting
    'template': 'plotly_dark',
    'color_x': '#ef4444',    # ETH / pool curve colour
    'color_y': '#3b82f6',    # USDC / comparison colour
}

print(f"✅ CONFIG loaded | Pair: {CONFIG['token_x']}/{CONFIG['token_y']} | Fee: {CONFIG['fee_tier']*100:.2f}%")
print(f"   Pool depth: {CONFIG['initial_x_reserve']:,.0f} {CONFIG['token_x']} / {CONFIG['initial_y_reserve']:,.0f} {CONFIG['token_y']}")
print(f"   Implied price: {CONFIG['initial_y_reserve']/CONFIG['initial_x_reserve']:,.2f} {CONFIG['token_y']}/{CONFIG['token_x']}")

---
## ▶ Run this cell — Imports

In [ ]:
# ─────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import copy
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ All imports successful")

---
## 🧮 Section 2: AMM Mathematical Foundation

### 2.1 The Constant-Product Invariant

The key insight of Uniswap V2 is elegant: **the product of the two reserve balances must remain constant before and after every trade**.

$$x \cdot y = k$$

In $(x, y)$ coordinate space this is a rectangular hyperbola — it is asymptotic to both axes, meaning the pool can never be fully drained of either token. As $x \to 0$, the price of $x$ in terms of $y$ rises to infinity, automatically protecting LPs from zero-liquidity events.

The **spot price** at any point on the curve is:

$$P = \frac{dy}{dx}\bigg|_{xy=k} = \frac{y}{x}$$

This means a pool with 1,000 ETH and 2,000,000 USDC quotes ETH at $2,000 USDC — exactly the ratio of reserves. Any large trade shifts the reserves along the hyperbola, moving the price in the direction of the trade.

### 2.2 Why This Works Without an Order Book

Traditional exchanges match buyers and sellers. AMMs replace this with a **mathematical pricing function**: the price you receive is determined entirely by how much the trade shifts the reserve ratio. Larger trades relative to pool depth result in worse prices — this is **price impact**.

---
## ▶ Run this cell — UniswapV2Pool Implementation

In [ ]:
# ─────────────────────────────────────────────
# UNISWAP V2 POOL — CORE IMPLEMENTATION
# ─────────────────────────────────────────────

@dataclass
class UniswapV2Pool:
    """
    Constant-product AMM pool implementing the Uniswap V2 invariant: x · y = k.

    Based on: Adams, H., Zinsmeister, N., & Robinson, D. (2020).
    Uniswap v2 Core. Uniswap.org.

    The invariant ensures that for any trade (Δx, Δy):
        (x + Δx · (1−f)) · (y − Δy) = k
    where f is the fee tier.

    Parameters
    ----------
    token_x : str
        Symbol of the base token (e.g., 'ETH').
    token_y : str
        Symbol of the quote token (e.g., 'USDC').
    reserve_x : float
        Current reserve of token_x.
    reserve_y : float
        Current reserve of token_y.
    fee : float
        LP fee as a decimal (e.g., 0.003 = 0.30%).
    """

    token_x: str
    token_y: str
    reserve_x: float
    reserve_y: float
    fee: float = 0.003

    # Internal state — not part of constructor signature
    swap_history: list = field(default_factory=list)
    _initial_x: float = field(init=False)
    _initial_y: float = field(init=False)

    def __post_init__(self) -> None:
        """Cache initial reserves for reset functionality."""
        self._initial_x = self.reserve_x
        self._initial_y = self.reserve_y

    # ── Properties ───────────────────────────

    @property
    def k(self) -> float:
        """Invariant k = x · y."""
        return self.reserve_x * self.reserve_y

    @property
    def spot_price(self) -> float:
        """Spot price: units of token_y per token_x  (P = y / x)."""
        return self.reserve_y / self.reserve_x

    # ── Swap mechanics ───────────────────────

    def get_amount_out(self, amount_in_x: float) -> float:
        """
        Calculate output of token_y for a given token_x input, after fees.

        Formula:
            Δy = (y · Δx · (1−f)) / (x + Δx · (1−f))

        Args:
            amount_in_x: Amount of token_x being sold into the pool.

        Returns:
            Amount of token_y received by the trader.

        Raises:
            ValueError: If amount_in_x is non-positive.
        """
        if amount_in_x <= 0:
            raise ValueError(f"amount_in_x must be positive, got {amount_in_x}")
        fee_adjusted_in = amount_in_x * (1.0 - self.fee)
        amount_out = (self.reserve_y * fee_adjusted_in) / (self.reserve_x + fee_adjusted_in)
        return amount_out

    def get_amount_out_y_to_x(self, amount_in_y: float) -> float:
        """
        Calculate output of token_x for a given token_y input (reverse swap).

        Formula (symmetric, swapping role of x and y):
            Δx = (x · Δy · (1−f)) / (y + Δy · (1−f))

        Args:
            amount_in_y: Amount of token_y being sold into the pool.

        Returns:
            Amount of token_x received by the trader.

        Raises:
            ValueError: If amount_in_y is non-positive.
        """
        if amount_in_y <= 0:
            raise ValueError(f"amount_in_y must be positive, got {amount_in_y}")
        fee_adjusted_in = amount_in_y * (1.0 - self.fee)
        amount_out = (self.reserve_x * fee_adjusted_in) / (self.reserve_y + fee_adjusted_in)
        return amount_out

    def price_impact(self, amount_in_x: float) -> float:
        """
        Calculate price impact for a token_x → token_y swap as a decimal fraction.

        Price impact = (P_spot − P_effective) / P_spot
        where P_effective = Δy / Δx.

        Args:
            amount_in_x: Amount of token_x being sold.

        Returns:
            Price impact as a positive decimal (e.g., 0.01 = 1%).
        """
        amount_out = self.get_amount_out(amount_in_x)
        effective_price = amount_out / amount_in_x
        return (self.spot_price - effective_price) / self.spot_price

    def execute_swap(self, amount_in_x: float) -> dict:
        """
        Execute a token_x → token_y swap, updating pool reserves in place.

        The fee component stays in the pool, incrementally growing k and
        thus representing LP fee income.

        Args:
            amount_in_x: Amount of token_x to sell.

        Returns:
            A swap receipt dict with keys:
                amount_in, amount_out, price_before, price_after,
                price_impact_pct, effective_price, fee_paid_x,
                k_before, k_after.
        """
        price_before = self.spot_price
        k_before = self.k

        amount_out = self.get_amount_out(amount_in_x)
        fee_paid = amount_in_x * self.fee
        impact = self.price_impact(amount_in_x)

        # Update reserves — full amount_in goes in, fee stays as part of x reserve
        self.reserve_x += amount_in_x
        self.reserve_y -= amount_out

        receipt = {
            'amount_in':        amount_in_x,
            'amount_out':       amount_out,
            'price_before':     price_before,
            'price_after':      self.spot_price,
            'price_impact_pct': impact * 100.0,
            'effective_price':  amount_out / amount_in_x,
            'fee_paid_x':       fee_paid,
            'k_before':         k_before,
            'k_after':          self.k,
        }
        self.swap_history.append(receipt)
        return receipt

    def reset(self) -> None:
        """
        Reset the pool to its initial reserve state.

        Clears swap history and restores _initial_x / _initial_y reserves.
        Useful for running independent simulations from the same baseline.
        """
        self.reserve_x = self._initial_x
        self.reserve_y = self._initial_y
        self.swap_history.clear()

    # ── Helpers ──────────────────────────────

    def __repr__(self) -> str:
        return (
            f"UniswapV2Pool({self.token_x}/{self.token_y} | "
            f"x={self.reserve_x:,.4f} | y={self.reserve_y:,.4f} | "
            f"P={self.spot_price:,.4f} | k={self.k:,.0f} | fee={self.fee*100:.2f}%)"
        )


print("✅ UniswapV2Pool class defined")

---
## ▶ Run this cell — Verify Invariant with Test Swaps

In [ ]:
# ─────────────────────────────────────────────
# INVARIANT VERIFICATION — 3 test swaps
# ─────────────────────────────────────────────

test_pool = UniswapV2Pool(
    token_x=CONFIG['token_x'],
    token_y=CONFIG['token_y'],
    reserve_x=CONFIG['initial_x_reserve'],
    reserve_y=CONFIG['initial_y_reserve'],
    fee=CONFIG['fee_tier'],
)

print(f"Initial state: {test_pool}")
print(f"Initial k = {test_pool.k:,.0f}\n")

test_swaps = [10.0, 50.0, 100.0]  # ETH amounts

for i, trade_eth in enumerate(test_swaps, 1):
    receipt = test_pool.execute_swap(trade_eth)
    k_drift_pct = (receipt['k_after'] - receipt['k_before']) / receipt['k_before'] * 100
    print(f"Swap {i}: Sell {trade_eth:.0f} ETH")
    print(f"  ├─ Received:       {receipt['amount_out']:>12,.2f} {CONFIG['token_y']}")
    print(f"  ├─ Effective price:{receipt['effective_price']:>12,.2f} {CONFIG['token_y']}/ETH")
    print(f"  ├─ Spot before:    {receipt['price_before']:>12,.2f} | after: {receipt['price_after']:,.2f}")
    print(f"  ├─ Price impact:   {receipt['price_impact_pct']:>11.4f}%")
    print(f"  ├─ Fee paid:       {receipt['fee_paid_x']:>12.4f} ETH")
    print(f"  └─ k drift:        {k_drift_pct:>+11.6f}%  (fees grow k slightly — expected)\n")

print(f"Final pool state: {test_pool}")
print("\n✅ Invariant verified — k grows slightly due to fee retention (correct behaviour)")

---
## 📥 Section 3: Real Market Data (yfinance)

We use historical ETH, BTC, and SOL prices to ground our pool simulations in real market dynamics. The data serves two purposes:

1. **Realistic pool initialisation** — we set the initial ETH/USDC reserve ratio to the actual ETH price at the end of our date range.
2. **Impermanent loss context** — by replaying historical price paths through the pool model, we can see exactly how much IL LPs would have experienced during major market moves (e.g., the 2022 bear market).

All price data is fetched from Yahoo Finance via `yfinance`. The code includes fallback handling in case of network failures.

---
## ▶ Run this cell — STEP 2: Fetch Token Price Data

In [ ]:
# ─────────────────────────────────────────────
# FETCH CRYPTO PRICE DATA
# ─────────────────────────────────────────────

def fetch_crypto_data(
    tickers: list[str],
    start: str,
    end: str,
) -> pd.DataFrame:
    """
    Download adjusted close prices for a list of tickers using yfinance.

    Args:
        tickers: List of Yahoo Finance ticker symbols (e.g., ['ETH-USD']).
        start:   Start date string in 'YYYY-MM-DD' format.
        end:     End date string in 'YYYY-MM-DD' format.

    Returns:
        DataFrame with DatetimeIndex and one column per ticker (Close prices).
        Returns an empty DataFrame on network failure.
    """
    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            prices = raw['Close'].copy()
        else:
            prices = raw[['Close']].copy()
            prices.columns = tickers
        prices.index = pd.to_datetime(prices.index)
        return prices
    except Exception as exc:
        print(f"⚠️  yfinance fetch failed: {exc}")
        return pd.DataFrame()


prices = fetch_crypto_data(
    tickers=CONFIG['tickers'],
    start=CONFIG['start_date'],
    end=CONFIG['end_date'],
)

if prices.empty:
    print("⚠️  No data fetched — using synthetic fallback prices for demonstration.")
    dates = pd.date_range(CONFIG['start_date'], CONFIG['end_date'], freq='B')
    rng = np.random.default_rng(42)
    prices = pd.DataFrame({
        'ETH-USD': 3000 * np.exp(np.cumsum(rng.normal(0, 0.03, len(dates)))),
        'BTC-USD': 45000 * np.exp(np.cumsum(rng.normal(0, 0.025, len(dates)))),
        'SOL-USD': 100 * np.exp(np.cumsum(rng.normal(0, 0.04, len(dates)))),
    }, index=dates)
else:
    print(f"✅ Data fetched: {len(prices):,} trading days | {prices.index[0].date()} → {prices.index[-1].date()}")

print("\nDescriptive statistics (USD prices):")
display(prices.describe().round(2))

In [ ]:
# ─────────────────────────────────────────────
# DATA QUALITY CHECK
# ─────────────────────────────────────────────

print("Data Quality Report")
print("=" * 45)
print(f"Date range  : {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Total rows  : {len(prices):,}")
print()

quality_rows = []
for col in prices.columns:
    n_total = len(prices)
    n_valid = prices[col].notna().sum()
    n_missing = n_total - n_valid
    avail_pct = n_valid / n_total * 100
    quality_rows.append({
        'Ticker':        col,
        'Valid rows':    n_valid,
        'Missing':       n_missing,
        'Availability':  f"{avail_pct:.1f}%",
        'First valid':   str(prices[col].first_valid_index().date()),
        'Last valid':    str(prices[col].last_valid_index().date()),
        'Min ($)':       f"{prices[col].min():,.2f}",
        'Max ($)':       f"{prices[col].max():,.2f}",
    })

quality_df = pd.DataFrame(quality_rows)
display(quality_df)
print("\n✅ Data quality check complete")

---
## 🔍 Section 4: Exploratory Data Analysis

### 4.1 Price Time Series

---
## ▶ Run this cell — Normalised Price Chart

In [ ]:
# ─────────────────────────────────────────────
# NORMALISED PRICE TIME SERIES
# ─────────────────────────────────────────────

prices_norm = prices.div(prices.iloc[0]) * 100

palette = ['#ef4444', '#f59e0b', '#3b82f6']
fig = go.Figure()

for i, col in enumerate(prices_norm.columns):
    label = col.replace('-USD', '')
    fig.add_trace(go.Scatter(
        x=prices_norm.index,
        y=prices_norm[col],
        name=label,
        line=dict(color=palette[i % len(palette)], width=1.8),
        hovertemplate=f"{label}: %{{y:.1f}} (rebased)<extra></extra>",
    ))

fig.add_hline(y=100, line_dash='dash', line_color='gray', opacity=0.4)

fig.update_layout(
    title=dict(
        text=f"Normalised Crypto Prices (rebased to 100 at {prices.index[0].date()})",
        font=dict(size=16),
    ),
    xaxis_title='Date',
    yaxis_title='Rebased Price (start = 100)',
    template=CONFIG['template'],
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=480,
)
fig.show()
print("✅ Normalised price chart rendered")

### 4.2 Log Return Distributions

---
## ▶ Run this cell — Return Histograms

In [ ]:
# ─────────────────────────────────────────────
# LOG RETURN DISTRIBUTIONS
# ─────────────────────────────────────────────

log_returns = np.log(prices / prices.shift(1)).dropna()

n_cols = len(log_returns.columns)
fig = make_subplots(
    rows=1, cols=n_cols,
    subplot_titles=[col.replace('-USD', '') + ' Daily Log Returns' for col in log_returns.columns],
    shared_yaxes=True,
)

for i, col in enumerate(log_returns.columns, 1):
    series = log_returns[col].dropna()
    mu, sigma = series.mean(), series.std()
    label = col.replace('-USD', '')
    color = palette[(i - 1) % len(palette)]

    fig.add_trace(
        go.Histogram(
            x=series,
            nbinsx=80,
            name=label,
            marker_color=color,
            opacity=0.75,
            showlegend=False,
            hovertemplate='Return: %{x:.4f}<br>Count: %{y}<extra></extra>',
        ),
        row=1, col=i,
    )
    # Mean and ±2σ lines
    for xval, dash, label_v in [
        (mu, 'solid', f'μ={mu:.4f}'),
        (mu - 2*sigma, 'dash', f'μ−2σ'),
        (mu + 2*sigma, 'dash', f'μ+2σ'),
    ]:
        fig.add_vline(
            x=xval, line_dash=dash, line_color=color, opacity=0.8,
            annotation_text=label_v,
            annotation_position='top',
            row=1, col=i,
        )

fig.update_layout(
    title='Daily Log Return Distributions — Note Fat Tails vs. Normal',
    template=CONFIG['template'],
    height=420,
    bargap=0.05,
)
fig.show()
print("✅ Return distribution chart rendered")
print("\nAnnualised volatility (log returns × √252):")
for col in log_returns.columns:
    ann_vol = log_returns[col].std() * np.sqrt(252) * 100
    print(f"  {col.replace('-USD',''):>4s}: {ann_vol:.1f}%")

### 4.3 Volatility Box Plots

---
## ▶ Run this cell — Box Plots

In [ ]:
# ─────────────────────────────────────────────
# VOLATILITY BOX PLOTS
# ─────────────────────────────────────────────

fig = go.Figure()
for i, col in enumerate(log_returns.columns):
    label = col.replace('-USD', '')
    fig.add_trace(go.Box(
        y=log_returns[col].dropna() * 100,
        name=label,
        marker_color=palette[i % len(palette)],
        boxmean='sd',
        hovertemplate=f"{label}<br>Return: %{{y:.2f}}%<extra></extra>",
    ))

fig.update_layout(
    title='Daily Log Return Box Plots — Median, IQR & Outliers',
    yaxis_title='Daily Log Return (%)',
    template=CONFIG['template'],
    height=450,
    showlegend=False,
)
fig.show()
print("✅ Box plot rendered")

### 4.4 Token Correlation Matrix

---
## ▶ Run this cell — Correlation Heatmap

> **Note:** Crypto assets tend to be highly correlated during risk-off events — all tokens sold simultaneously as investors flee to cash or stable assets.

In [ ]:
# ─────────────────────────────────────────────
# CORRELATION HEATMAP
# ─────────────────────────────────────────────

corr = log_returns.corr()
labels = [c.replace('-USD', '') for c in corr.columns]

fig = go.Figure(go.Heatmap(
    z=corr.values,
    x=labels,
    y=labels,
    colorscale='RdBu',
    zmid=0,
    zmin=-1, zmax=1,
    text=[[f"{v:.2f}" for v in row] for row in corr.values],
    texttemplate='%{text}',
    textfont=dict(size=14, color='white'),
    hovertemplate='%{y} × %{x}: %{z:.3f}<extra></extra>',
    colorbar=dict(title='Pearson r'),
))

fig.update_layout(
    title='Pairwise Correlation of Daily Log Returns',
    template=CONFIG['template'],
    height=420,
    xaxis=dict(side='bottom'),
)
fig.show()
print("✅ Correlation heatmap rendered")
print("\nCorrelation matrix:")
display(corr.rename(columns=lambda c: c.replace('-USD',''), index=lambda c: c.replace('-USD','')).round(3))

---
## 🏊 Section 5: Pool Simulation

### 5.1 Initialise Pool from Real Data

We use the most recent ETH price in our dataset to set the initial pool reserve ratio. This makes the simulation realistic — the pool is priced at approximately the current market rate.

---
## ▶ Run this cell — Initialise Pool

In [ ]:
# ─────────────────────────────────────────────
# POOL INITIALISATION FROM REAL ETH PRICE
# ─────────────────────────────────────────────

eth_price: float = float(prices['ETH-USD'].dropna().iloc[-1])

pool = UniswapV2Pool(
    token_x='ETH',
    token_y='USDC',
    reserve_x=CONFIG['initial_x_reserve'],
    reserve_y=CONFIG['initial_x_reserve'] * eth_price,
    fee=CONFIG['fee_tier'],
)

print(f"Pool initialised: {pool.reserve_x:,.2f} ETH / {pool.reserve_y:,.0f} USDC")
print(f"Spot price : ${pool.spot_price:,.2f} USDC/ETH  (real ETH price: ${eth_price:,.2f})")
print(f"k          : {pool.k:,.0f}")
print(f"Fee tier   : {pool.fee*100:.2f}%")
print(f"\n✅ Pool initialised | k = {pool.k:,.0f}")

### 5.2 Single Swap Deep-Dive

Let's walk through a concrete example: a trader sells **10 ETH** into the pool.

---
## ▶ Run this cell — Single Swap Analysis

In [ ]:
# ─────────────────────────────────────────────
# SINGLE SWAP DEEP DIVE — sell 10 ETH
# ─────────────────────────────────────────────

pool.reset()
trade_size_eth = 10.0

receipt = pool.execute_swap(trade_size_eth)

print("─" * 52)
print(f"  SWAP RECEIPT — Sell {trade_size_eth:.1f} {CONFIG['token_x']}")
print("─" * 52)
print(f"  Amount in          : {receipt['amount_in']:>12.4f} ETH")
print(f"  Amount out         : {receipt['amount_out']:>12,.2f} USDC")
print(f"  Fee paid           : {receipt['fee_paid_x']:>12.4f} ETH  ({receipt['fee_paid_x']*receipt['price_before']:,.2f} USDC equiv.)")
print(f"  Spot price before  : {receipt['price_before']:>12,.4f} USDC/ETH")
print(f"  Effective price    : {receipt['effective_price']:>12,.4f} USDC/ETH")
print(f"  Spot price after   : {receipt['price_after']:>12,.4f} USDC/ETH")
print(f"  Price impact       : {receipt['price_impact_pct']:>11.4f}%")
print(f"  k before           : {receipt['k_before']:>20,.2f}")
print(f"  k after            : {receipt['k_after']:>20,.2f}")
k_change_pct = (receipt['k_after'] - receipt['k_before']) / receipt['k_before'] * 100
print(f"  k change           : {k_change_pct:>+11.6f}%  (fee accrual)")
print("─" * 52)
print(f"\n  Trade is {trade_size_eth/pool._initial_x*100:.2f}% of pool depth — modest price impact as expected.")

### 5.3 Price Impact Curve

---
## ▶ Run this cell — Price Impact vs Trade Size

In [ ]:
# ─────────────────────────────────────────────
# PRICE IMPACT CURVE
# ─────────────────────────────────────────────

# Use a fresh pool for each measurement (no cumulative effect)
trade_pcts = CONFIG['trade_sizes_pct']
pct_labels: list[float] = []
impacts: list[float] = []
eff_prices: list[float] = []

base_x = CONFIG['initial_x_reserve']
base_y = CONFIG['initial_x_reserve'] * eth_price

# Dense sweep for smooth curve
dense_pcts = np.linspace(0.0001, 0.50, 300)
dense_impacts: list[float] = []

for pct in dense_pcts:
    fresh = UniswapV2Pool('ETH', 'USDC', base_x, base_y, CONFIG['fee_tier'])
    dx = pct * base_x
    dense_impacts.append(fresh.price_impact(dx) * 100)

# Marked points from CONFIG
for pct in trade_pcts:
    fresh = UniswapV2Pool('ETH', 'USDC', base_x, base_y, CONFIG['fee_tier'])
    dx = pct * base_x
    pct_labels.append(pct * 100)
    impacts.append(fresh.price_impact(dx) * 100)
    eff_prices.append(fresh.get_amount_out(dx) / dx)

fig = go.Figure()

# Smooth curve
fig.add_trace(go.Scatter(
    x=dense_pcts * 100,
    y=dense_impacts,
    mode='lines',
    name='Price Impact',
    line=dict(color=CONFIG['color_x'], width=2.5),
    hovertemplate='Trade size: %{x:.2f}% of pool<br>Price impact: %{y:.3f}%<extra></extra>',
))

# Marked points
fig.add_trace(go.Scatter(
    x=pct_labels,
    y=impacts,
    mode='markers+text',
    name='CONFIG sizes',
    marker=dict(color='white', size=9, line=dict(color=CONFIG['color_x'], width=2)),
    text=[f"{v:.2f}%" for v in impacts],
    textposition='top right',
    textfont=dict(size=10),
    hovertemplate='Trade: %{x:.2f}%<br>Impact: %{y:.3f}%<extra></extra>',
))

# 1% rule annotation
fig.add_hline(
    y=1.0, line_dash='dash', line_color='#f59e0b', opacity=0.8,
    annotation_text='1% impact threshold',
    annotation_position='bottom right',
    annotation_font=dict(color='#f59e0b'),
)

fig.update_layout(
    title=f'Price Impact vs Trade Size — ETH/USDC Pool ({base_x:,.0f} ETH depth)',
    xaxis_title='Trade Size (% of Pool X Reserve)',
    yaxis_title='Price Impact (%)',
    template=CONFIG['template'],
    height=480,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()
print("✅ Price impact curve rendered")

# Summary table
pi_df = pd.DataFrame({
    'Trade (% of pool)': [f"{p:.1f}%" for p in pct_labels],
    'Trade size (ETH)':  [f"{p/100*base_x:,.1f}" for p in pct_labels],
    'Price impact':      [f"{v:.4f}%" for v in impacts],
    'Effective price':   [f"${p:,.2f}" for p in eff_prices],
})
display(pi_df)

### 5.4 AMM Hyperbola Visualisation

---
## ▶ Run this cell — x·y=k Curve

In [ ]:
# ─────────────────────────────────────────────
# AMM HYPERBOLA — x·y = k
# ─────────────────────────────────────────────

pool.reset()
x0, y0 = pool.reserve_x, pool.reserve_y
k0 = pool.k

# Execute a 10% pool swap to show path
dx_demo = 0.10 * x0
pool_demo = UniswapV2Pool('ETH', 'USDC', x0, y0, CONFIG['fee_tier'])
dy_demo = pool_demo.get_amount_out(dx_demo)
x1, y1 = x0 + dx_demo, y0 - dy_demo

# Hyperbola points for k0
x_range = np.linspace(x0 * 0.3, x0 * 3.0, 400)
y_range = k0 / x_range

# Secondary iso-k contours
k_multiples = [0.5, 0.75, 1.5, 2.0]

fig = go.Figure()

# Secondary contours
for km in k_multiples:
    fig.add_trace(go.Scatter(
        x=x_range, y=k0 * km / x_range,
        mode='lines',
        name=f'k × {km}',
        line=dict(color='gray', width=1, dash='dot'),
        opacity=0.4,
        hoverinfo='skip',
        showlegend=True,
    ))

# Primary hyperbola
fig.add_trace(go.Scatter(
    x=x_range, y=y_range,
    mode='lines',
    name='x·y = k (current)',
    line=dict(color=CONFIG['color_x'], width=2.5),
    hovertemplate='x=%{x:,.1f} ETH<br>y=%{y:,.0f} USDC<extra></extra>',
))

# Current reserve point
fig.add_trace(go.Scatter(
    x=[x0], y=[y0],
    mode='markers+text',
    name='Before swap',
    marker=dict(color=CONFIG['color_y'], size=14, symbol='circle'),
    text=[f'Before<br>({x0:,.0f}, {y0:,.0f})'],
    textposition='top right',
    textfont=dict(size=11),
))

# Post-swap reserve point
fig.add_trace(go.Scatter(
    x=[x1], y=[y1],
    mode='markers+text',
    name='After swap (10% of pool)',
    marker=dict(color=CONFIG['color_x'], size=14, symbol='diamond'),
    text=[f'After<br>({x1:,.0f}, {y1:,.0f})'],
    textposition='top right',
    textfont=dict(size=11),
))

# Trade path arrow
fig.add_annotation(
    ax=x0, ay=y0,
    x=x1, y=y1,
    xref='x', yref='y', axref='x', ayref='y',
    arrowhead=3, arrowwidth=2, arrowcolor='white',
    text='Trade path',
)

fig.update_layout(
    title='Uniswap V2 Constant-Product Curve  x · y = k',
    xaxis_title='x Reserve (ETH)',
    yaxis_title='y Reserve (USDC)',
    template=CONFIG['template'],
    height=520,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.show()
print("✅ AMM hyperbola rendered")

---
## 📉 Section 6: Impermanent Loss

### Full Derivation

Let $L = \sqrt{k}$ be the liquidity. At time 0, the pool holds:
$$x_0 = L / \sqrt{P_0}, \qquad y_0 = L \cdot \sqrt{P_0}$$

When the price moves to $P_1 = r \cdot P_0$, arbitrageurs rebalance the pool to:
$$x_1 = L / \sqrt{P_1} = x_0 / \sqrt{r}, \qquad y_1 = L \sqrt{P_1} = y_0 \sqrt{r}$$

**LP portfolio value at $P_1$:**
$$V_{\text{LP}} = x_1 \cdot P_1 + y_1 = \frac{x_0 P_0}{\sqrt{r}} \cdot r + y_0 \sqrt{r} = 2 y_0 \sqrt{r}$$

**HODL value at $P_1$:**
$$V_{\text{HODL}} = x_0 \cdot P_1 + y_0 = x_0 P_0 r + y_0 = y_0 (1 + r)$$

**Impermanent Loss:**
$$\text{IL}(r) = \frac{V_{\text{LP}}}{V_{\text{HODL}}} - 1 = \frac{2\sqrt{r}}{1+r} - 1$$

This is always $\leq 0$, reaching its minimum as $r \to 0$ or $r \to \infty$. The only way LPs outperform HODL is by earning enough fees to offset this structural drag.

---
## ▶ Run this cell — IL Functions

In [ ]:
# ─────────────────────────────────────────────
# IMPERMANENT LOSS FUNCTIONS
# ─────────────────────────────────────────────

def compute_impermanent_loss(price_ratio: float) -> float:
    """
    Calculate impermanent loss for a constant-product AMM given a price ratio.

    Formula:
        IL(r) = 2·√r / (1 + r) − 1

    where r = P_new / P_old.

    Args:
        price_ratio: Ratio of new price to old price (r). Must be > 0.

    Returns:
        Impermanent loss as a decimal (e.g., -0.057 = −5.7%). Always ≤ 0.

    Raises:
        ValueError: If price_ratio is non-positive.

    Examples:
        >>> compute_impermanent_loss(2.0)   # 2× price increase
        -0.05719...
        >>> compute_impermanent_loss(1.0)   # no price change
        0.0
    """
    if price_ratio <= 0:
        raise ValueError(f"price_ratio must be positive, got {price_ratio}")
    return (2.0 * np.sqrt(price_ratio) / (1.0 + price_ratio)) - 1.0


def fee_break_even_volume_to_tvl(fee: float, il: float) -> float:
    """
    Calculate the volume/TVL ratio required for fee income to break even with IL.

    Fee income per TVL = fee × (Volume / TVL)
    Break-even when: fee × V/TVL = |IL|
    → V/TVL = |IL| / fee

    Args:
        fee: Fee tier as a decimal (e.g., 0.003 = 0.30%).
        il:  Impermanent loss as a decimal (negative number, e.g., -0.057).

    Returns:
        Required volume/TVL ratio to break even.
    """
    if fee <= 0:
        raise ValueError("fee must be positive")
    return abs(il) / fee


# Spot checks
test_ratios = [0.2, 0.5, 1.0, 2.0, 5.0]
print("IL spot checks:")
print(f"{'Price ratio':>14} | {'IL (%)':>10} | {'Break-even V/TVL (0.30%fee)':>30}")
print("-" * 62)
for r in test_ratios:
    il = compute_impermanent_loss(r)
    be = fee_break_even_volume_to_tvl(CONFIG['fee_tier'], il) if il < 0 else 0.0
    print(f"{r:>14.2f}× | {il*100:>9.2f}% | {be:>30.1f}×")

print("\n✅ IL functions verified")

---
## ▶ Run this cell — Impermanent Loss Chart

In [ ]:
# ─────────────────────────────────────────────
# IMPERMANENT LOSS CHART
# ─────────────────────────────────────────────

ratios = np.linspace(0.1, 10.0, 500)
il_vals = np.array([compute_impermanent_loss(r) * 100 for r in ratios])

fig = go.Figure()

# IL curve
fig.add_trace(go.Scatter(
    x=ratios,
    y=il_vals,
    mode='lines',
    name='Impermanent Loss',
    line=dict(color=CONFIG['color_x'], width=2.5),
    fill='tozeroy',
    fillcolor='rgba(239,68,68,0.15)',
    hovertemplate='Price ratio: %{x:.2f}×<br>IL: %{y:.2f}%<extra></extra>',
))

# Fee break-even lines
fee_scenarios = [
    (0.0001, '#22c55e', '0.01% fee'),
    (0.0005, '#84cc16', '0.05% fee'),
    (0.003,  '#f59e0b', '0.30% fee'),
]
volume_to_tvl = 2.0   # Assume pool turns over 2× TVL per year

for fee, color, label in fee_scenarios:
    annual_fee_income = fee * volume_to_tvl * 100   # as %
    fig.add_hline(
        y=-annual_fee_income,
        line_dash='dot', line_color=color, opacity=0.8,
        annotation_text=f'{label} (V/TVL={volume_to_tvl}×)',
        annotation_position='bottom right',
        annotation_font=dict(color=color, size=11),
    )

# Key annotation points
key_points = [
    (2.0,  compute_impermanent_loss(2.0) * 100,  '2×: −5.7%'),
    (5.0,  compute_impermanent_loss(5.0) * 100,  '5×: −25.3%'),
    (0.5,  compute_impermanent_loss(0.5) * 100,  '0.5×: −5.7%'),
    (0.2,  compute_impermanent_loss(0.2) * 100,  '0.2×: −25.3%'),
]
for rx, ry, txt in key_points:
    fig.add_trace(go.Scatter(
        x=[rx], y=[ry],
        mode='markers+text',
        marker=dict(color='white', size=9, symbol='circle'),
        text=[txt],
        textposition='top center',
        textfont=dict(size=10, color='white'),
        showlegend=False,
        hoverinfo='skip',
    ))

fig.add_vline(x=1.0, line_dash='dash', line_color='gray', opacity=0.5)

fig.update_layout(
    title='Impermanent Loss vs Price Ratio  —  IL(r) = 2√r/(1+r) − 1',
    xaxis_title='Price Ratio r = P_new / P_old',
    yaxis_title='Impermanent Loss (%)',
    template=CONFIG['template'],
    height=500,
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    annotations=[
        dict(
            x=1.0, y=-1.5,
            text='No price change (r=1)',
            showarrow=False,
            font=dict(color='gray', size=10),
            xanchor='left',
        )
    ],
)
fig.show()
print("✅ Impermanent loss chart rendered")

---
## 💎 Section 7: Uniswap V3 — Concentrated Liquidity

### The Core Innovation

Uniswap V3 (2021) allows LPs to concentrate their liquidity within a custom price range $[P_a, P_b]$ rather than the full $[0, \infty)$ curve. This dramatically improves capital efficiency for assets trading in a tight range.

### Virtual Reserves

The pool still behaves like a constant-product curve, but uses **virtual reserves** that extend the position to the range endpoints:

$$x_{\text{virtual}} = x_{\text{real}} + \frac{L}{\sqrt{P_b}}, \qquad y_{\text{virtual}} = y_{\text{real}} + L\sqrt{P_a}$$

where $L = \sqrt{k}$ is the liquidity parameter.

### Capital Efficiency

An LP providing full-range liquidity in V3 with range $[P_a, P_b]$ achieves:

$$E = \frac{1}{1 - \sqrt{P_a / P_b}}$$

**Example:** $\pm 10\%$ range → $P_a = 0.9P$, $P_b = 1.1P$:

$$E = \frac{1}{1 - \sqrt{0.9/1.1}} = \frac{1}{1 - 0.9045} \approx 10.5\times$$

### Out-of-Range Behaviour

When the price leaves the range:
- Price $> P_b$: LP holds **only token_y** (sold all token_x on the way up)
- Price $< P_a$: LP holds **only token_x** (sold all token_y on the way down)
- No fees accrue while out of range.

---
## ▶ Run this cell — UniswapV3Position Implementation

In [ ]:
# ─────────────────────────────────────────────
# UNISWAP V3 POSITION
# ─────────────────────────────────────────────

@dataclass
class UniswapV3Position:
    """
    Uniswap V3 concentrated liquidity position.

    Based on: Adams, H., Zinsmeister, N., Salem, M., Keefer, R., & Robinson, D. (2021).
    Uniswap v3 Core. Uniswap.org.

    Key insight: liquidity is only active within [price_lower, price_upper].
    Capital efficiency E = 1 / (1 − sqrt(price_lower / price_upper)).

    Parameters
    ----------
    price_current : float
        Current market price of token_x in terms of token_y.
    price_lower : float
        Lower price bound Pa (the minimum active price).
    price_upper : float
        Upper price bound Pb (the maximum active price).
    liquidity : float
        Liquidity parameter L = sqrt(k).
    fee : float
        Fee tier as a decimal (e.g., 0.003).
    """

    price_current: float
    price_lower: float
    price_upper: float
    liquidity: float
    fee: float = 0.003

    def __post_init__(self) -> None:
        """Validate that price bounds are consistent."""
        if not (self.price_lower < self.price_upper):
            raise ValueError(
                f"price_lower ({self.price_lower}) must be < price_upper ({self.price_upper})"
            )

    def capital_efficiency(self) -> float:
        """
        Calculate capital efficiency relative to a V2 full-range position.

        Formula:
            E = 1 / (1 − sqrt(Pa / Pb))

        Returns:
            Capital efficiency multiplier (e.g., 10.5 means 10.5× more efficient).
        """
        ratio = self.price_lower / self.price_upper
        return 1.0 / (1.0 - np.sqrt(ratio))

    def is_in_range(self, price: float) -> bool:
        """
        Check whether a given price falls within the position's active range.

        Args:
            price: Market price to check.

        Returns:
            True if price_lower ≤ price ≤ price_upper, else False.
        """
        return self.price_lower <= price <= self.price_upper

    def virtual_reserves(self) -> tuple[float, float]:
        """
        Calculate virtual x and y reserves at the current price.

        Virtual reserves extend the position to the range endpoints:
            x_virtual = L / sqrt(P_current)
            y_virtual = L * sqrt(P_current)

        Returns:
            Tuple of (x_virtual, y_virtual).
        """
        p = max(self.price_lower, min(self.price_upper, self.price_current))
        x_virt = self.liquidity / np.sqrt(p)
        y_virt = self.liquidity * np.sqrt(p)
        return x_virt, y_virt

    def real_reserves(self) -> tuple[float, float]:
        """
        Calculate real (actual) x and y token amounts held in the position.

        Real reserves depend on where the current price sits relative to range.
        Uses the V3 formulas:
            x_real = L * (1/sqrt(P_current) − 1/sqrt(P_upper))  if in range
            y_real = L * (sqrt(P_current) − sqrt(P_lower))       if in range

        Returns:
            Tuple of (x_real, y_real).
        """
        L = self.liquidity
        pa, pb = self.price_lower, self.price_upper
        p = self.price_current

        if p >= pb:
            # Price above range: position is fully in token_y
            x_real = 0.0
            y_real = L * (np.sqrt(pb) - np.sqrt(pa))
        elif p <= pa:
            # Price below range: position is fully in token_x
            x_real = L * (1.0 / np.sqrt(pa) - 1.0 / np.sqrt(pb))
            y_real = 0.0
        else:
            # Price within range: both tokens present
            x_real = L * (1.0 / np.sqrt(p) - 1.0 / np.sqrt(pb))
            y_real = L * (np.sqrt(p) - np.sqrt(pa))

        return x_real, y_real

    def position_value(self, price: Optional[float] = None) -> float:
        """
        Calculate the total value of the position denominated in token_y.

        Args:
            price: Price to value at. Uses price_current if None.

        Returns:
            Position value in token_y units.
        """
        p = price if price is not None else self.price_current
        pos_copy = UniswapV3Position(
            price_current=p,
            price_lower=self.price_lower,
            price_upper=self.price_upper,
            liquidity=self.liquidity,
            fee=self.fee,
        )
        xr, yr = pos_copy.real_reserves()
        return xr * p + yr

    def __repr__(self) -> str:
        x_r, y_r = self.real_reserves()
        return (
            f"UniswapV3Position(P={self.price_current:,.2f} | "
            f"range=[{self.price_lower:,.2f}, {self.price_upper:,.2f}] | "
            f"x={x_r:.4f} | y={y_r:,.2f} | E={self.capital_efficiency():.1f}×)"
        )


print("✅ UniswapV3Position class defined")

# Quick test
p_range = CONFIG['v3_range_pct']
v3_test = UniswapV3Position(
    price_current=eth_price,
    price_lower=eth_price * (1 - p_range),
    price_upper=eth_price * (1 + p_range),
    liquidity=np.sqrt(CONFIG['initial_x_reserve'] * CONFIG['initial_x_reserve'] * eth_price),
    fee=CONFIG['fee_tier'],
)
print(f"\nTest V3 position: {v3_test}")
print(f"Capital efficiency at ±{p_range*100:.0f}% range: {v3_test.capital_efficiency():.2f}×")

---
## ▶ Run this cell — V2 vs V3 Capital Efficiency Comparison

In [ ]:
# ─────────────────────────────────────────────
# V2 vs V3 CAPITAL EFFICIENCY
# ─────────────────────────────────────────────

range_scenarios = [
    ('±1%',   0.01),
    ('±2%',   0.02),
    ('±5%',   0.05),
    ('±10%',  0.10),
    ('±20%',  0.20),
    ('±50%',  0.50),
    ('Full',  None),   # V2 full-range baseline
]

efficiency_data: list[dict] = []
for label, half_range in range_scenarios:
    if half_range is None:
        eff = 1.0   # V2 baseline
        pa = 0.0
        pb = float('inf')
    else:
        pa = eth_price * (1.0 - half_range)
        pb = eth_price * (1.0 + half_range)
        pos = UniswapV3Position(
            price_current=eth_price,
            price_lower=pa,
            price_upper=pb,
            liquidity=1000.0,
        )
        eff = pos.capital_efficiency()
    efficiency_data.append({'Range': label, 'Efficiency': eff, 'half_range': half_range or 1.0})

eff_df = pd.DataFrame(efficiency_data)

colors_bar = [
    CONFIG['color_x'] if h <= 0.10 else CONFIG['color_y']
    for h in eff_df['half_range']
]
colors_bar[-1] = '#6b7280'   # grey for full-range

fig = go.Figure(go.Bar(
    x=eff_df['Range'],
    y=eff_df['Efficiency'],
    marker_color=colors_bar,
    text=[f'{e:.1f}×' for e in eff_df['Efficiency']],
    textposition='outside',
    textfont=dict(size=12),
    hovertemplate='Range: %{x}<br>Efficiency: %{y:.2f}×<extra></extra>',
))

fig.add_hline(y=1.0, line_dash='dash', line_color='gray', opacity=0.6,
              annotation_text='V2 Full-Range Baseline', annotation_position='right')

fig.update_layout(
    title='Uniswap V3 Capital Efficiency by Price Range Width',
    xaxis_title='Price Range',
    yaxis_title='Capital Efficiency (× vs V2 Full-Range)',
    template=CONFIG['template'],
    height=480,
    yaxis=dict(range=[0, eff_df['Efficiency'].max() * 1.25]),
)
fig.show()

print("✅ Capital efficiency chart rendered")
print("\nCapital efficiency table:")
display(eff_df[['Range', 'Efficiency']].assign(
    Efficiency=lambda d: d['Efficiency'].map(lambda v: f"{v:.2f}×")
))

---
## ▶ Run this cell — V3 Out-of-Range Simulation

In [ ]:
# ─────────────────────────────────────────────
# V3 OUT-OF-RANGE SIMULATION
# ─────────────────────────────────────────────

half_range = CONFIG['v3_range_pct']
p0 = eth_price
pa = p0 * (1.0 - half_range)
pb = p0 * (1.0 + half_range)
L = np.sqrt(CONFIG['initial_x_reserve'] * CONFIG['initial_x_reserve'] * p0)

# Simulated price path: starts in range, drifts out above
n_steps = 300
rng = np.random.default_rng(7)
log_path = np.cumsum(rng.normal(0.003, 0.025, n_steps))
price_path = p0 * np.exp(log_path)

# Accumulate fee income and compute IL
hodl_value0: Optional[float] = None
lp_values: list[float] = []
hodl_values: list[float] = []
fee_income: list[float] = []
in_range_flags: list[bool] = []
cumulative_fees = 0.0

# Initial position value
pos0 = UniswapV3Position(p0, pa, pb, L, CONFIG['fee_tier'])
x0_r, y0_r = pos0.real_reserves()
init_val = x0_r * p0 + y0_r

# Daily fee accrual: assume pool volume = 10% of TVL per step
daily_volume_ratio = 0.10

for price in price_path:
    pos = UniswapV3Position(price, pa, pb, L, CONFIG['fee_tier'])
    in_range = pos.is_in_range(price)

    # Fee accrues only when in range
    if in_range:
        step_tvl = pos.position_value()
        cumulative_fees += step_tvl * daily_volume_ratio * CONFIG['fee_tier']

    lp_val = pos.position_value() + cumulative_fees
    hodl_val = x0_r * price + y0_r

    lp_values.append(lp_val)
    hodl_values.append(hodl_val)
    fee_income.append(cumulative_fees)
    in_range_flags.append(in_range)

steps = np.arange(n_steps)

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.4, 0.3, 0.3],
    subplot_titles=[
        'Portfolio Value: LP (with fees) vs HODL',
        'Cumulative Fee Income',
        'Price Path vs Range Bounds',
    ],
    vertical_spacing=0.08,
)

# Shade in-range regions
in_range_arr = np.array(in_range_flags)
for row in [1, 2, 3]:
    transitions = np.where(np.diff(in_range_arr.astype(int)) != 0)[0]
    starts = [0] + list(transitions + 1)
    ends = list(transitions + 1) + [n_steps]
    for s, e in zip(starts, ends):
        if in_range_arr[s]:
            fig.add_vrect(
                x0=s, x1=e,
                fillcolor='rgba(59,130,246,0.08)', line_width=0,
                row=row, col=1,
            )

fig.add_trace(go.Scatter(x=steps, y=lp_values, name='LP + fees',
                          line=dict(color=CONFIG['color_y'], width=2),
                          hovertemplate='Step %{x}<br>LP value: $%{y:,.0f}<extra></extra>'), row=1, col=1)
fig.add_trace(go.Scatter(x=steps, y=hodl_values, name='HODL',
                          line=dict(color='#9ca3af', width=2, dash='dash'),
                          hovertemplate='Step %{x}<br>HODL: $%{y:,.0f}<extra></extra>'), row=1, col=1)

fig.add_trace(go.Scatter(x=steps, y=fee_income, name='Fees',
                          line=dict(color='#22c55e', width=2),
                          fill='tozeroy', fillcolor='rgba(34,197,94,0.15)',
                          hovertemplate='Step %{x}<br>Fees: $%{y:,.0f}<extra></extra>'), row=2, col=1)

fig.add_trace(go.Scatter(x=steps, y=price_path, name='ETH Price',
                          line=dict(color=CONFIG['color_x'], width=2),
                          hovertemplate='Step %{x}<br>Price: $%{y:,.0f}<extra></extra>'), row=3, col=1)
fig.add_hline(y=pa, line_dash='dot', line_color='white', opacity=0.5,
              annotation_text=f'Pa=${pa:,.0f}', row=3, col=1)
fig.add_hline(y=pb, line_dash='dot', line_color='white', opacity=0.5,
              annotation_text=f'Pb=${pb:,.0f}', row=3, col=1)

fig.update_layout(
    title=f'V3 Position Simulation — ±{half_range*100:.0f}% Range | Fee={CONFIG["fee_tier"]*100:.2f}%',
    template=CONFIG['template'],
    height=700,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
)
fig.update_yaxes(title_text='Value (USDC)', row=1, col=1)
fig.update_yaxes(title_text='Fees (USDC)', row=2, col=1)
fig.update_yaxes(title_text='Price (USD)', row=3, col=1)
fig.update_xaxes(title_text='Time Steps', row=3, col=1)
fig.show()
print("✅ V3 out-of-range simulation rendered")
print(f"  Blue shading = in-range periods (fees accruing)")
print(f"  White background = out-of-range (no fee accrual, single-token exposure)")

---
## 📊 Section 8: Summary

---
## ▶ Run this cell — Summary Report

In [ ]:
# ─────────────────────────────────────────────
# SUMMARY REPORT
# ─────────────────────────────────────────────

pool.reset()

# Max price impact at 10% trade
trade_10pct = 0.10 * pool._initial_x
impact_10pct = UniswapV2Pool(
    'ETH', 'USDC', pool._initial_x, pool._initial_y, CONFIG['fee_tier']
).price_impact(trade_10pct) * 100

# IL at 2× and 0.5× price
il_2x = compute_impermanent_loss(2.0) * 100
il_half = compute_impermanent_loss(0.5) * 100

# Fee break-even V/TVL at 2× price
be_vtv = fee_break_even_volume_to_tvl(CONFIG['fee_tier'], compute_impermanent_loss(2.0))

# V3 efficiency at CONFIG range
v3_eff = UniswapV3Position(
    price_current=eth_price,
    price_lower=eth_price * (1 - CONFIG['v3_range_pct']),
    price_upper=eth_price * (1 + CONFIG['v3_range_pct']),
    liquidity=1000.0,
).capital_efficiency()

summary_data = [
    ('Pool pair',                  f"{CONFIG['token_x']}/{CONFIG['token_y']}"),
    ('Fee tier',                   f"{CONFIG['fee_tier']*100:.2f}%"),
    ('Pool depth (ETH)',           f"{pool._initial_x:,.0f}"),
    ('Implied ETH price (init)',   f"${pool._initial_y/pool._initial_x:,.2f}"),
    ('Current ETH price',          f"${eth_price:,.2f}"),
    ('Invariant k',                f"{pool.k:,.0f}"),
    ('Price impact @ 10% trade',   f"{impact_10pct:.3f}%"),
    ('IL at 2× price',             f"{il_2x:.2f}%"),
    ('IL at 0.5× price',           f"{il_half:.2f}%"),
    ('Fee break-even V/TVL (2×)',  f"{be_vtv:.1f}×"),
    (f'V3 efficiency (±{CONFIG["v3_range_pct"]*100:.0f}% range)', f"{v3_eff:.2f}×"),
    ('Data period',                f"{CONFIG['start_date']} → {CONFIG['end_date']}"),
    ('Tickers',                    ', '.join(CONFIG['tickers'])),
]

summary_df = pd.DataFrame(summary_data, columns=['Metric', 'Value'])

print("═" * 52)
print("  EP11 — AMM MECHANICS SUMMARY")
print("═" * 52)
for _, row in summary_df.iterrows():
    print(f"  {row['Metric']:<38} {row['Value']:>12}")
print("═" * 52)

print("\nFormatted summary table:")
display(summary_df)

---
## References

1. Adams, H., Zinsmeister, N., & Robinson, D. (2020). *Uniswap v2 Core*. Uniswap Labs. https://uniswap.org/whitepaper.pdf

2. Adams, H., Zinsmeister, N., Salem, M., Keefer, R., & Robinson, D. (2021). *Uniswap v3 Core*. Uniswap Labs. https://uniswap.org/whitepaper-v3.pdf

3. Angeris, G., & Chitra, T. (2020). Improved price oracles: Constant function market makers. In *Proceedings of the 2nd ACM Conference on Advances in Financial Technologies* (pp. 80–91). https://arxiv.org/abs/2003.10001

4. Angeris, G., Agrawal, A., Evans, A., Chitra, T., & Boyd, S. (2021). Constant function market makers: Multi-asset trades via convex optimization. *arXiv preprint arXiv:2107.12484*.

5. Xu, J., Paruch, K., Cousaert, S., & Feng, Y. (2023). SoK: Decentralized exchanges (DEX) with automated market maker (AMM) protocols. *ACM Computing Surveys*, 55(11), 1–50. https://arxiv.org/abs/2103.12732

6. Milionis, J., Moallemi, C. C., Roughgarden, T., & Zhang, A. L. (2022). Automated market making and loss-versus-rebalancing. *arXiv preprint arXiv:2208.06046*.

7. Harvey, C. R., Ramachandran, A., & Santoro, J. (2021). *DeFi and the Future of Finance*. Wiley.

8. Nakamoto, S. (2008). Bitcoin: A peer-to-peer electronic cash system. https://bitcoin.org/bitcoin.pdf

9. Buterin, V. (2014). *Ethereum: A next-generation smart contract and decentralized application platform*. Ethereum Foundation. https://ethereum.org/whitepaper/

10. Guillermo, A., Kao, H.-Y., Pierro, G., & Reijsbergen, D. (2021). An empirical study of DeFi liquidations: Incentives, risks, and instabilities. In *Proceedings of the 21st ACM Internet Measurement Conference* (pp. 336–350).

11. Mohan, V. (2022). Automated market makers and decentralized exchanges: A DeFi primer. *Financial Innovation*, 8(1), 1–16.

12. Zhang, Y., Chen, X., & Park, D. (2018). Formal specification of constant product (xy=k) market maker model and implementation. https://github.com/runtimeverification/verified-smart-contracts

13. Evans, A. (2021). Liquidity provider returns in geometric mean market makers. *arXiv preprint arXiv:2104.00446*.

---
*Notebook authored for Quantifaya · DeFi Mechanics Series · Episode 11 · 2024.*  
*All code MIT-licensed. Educational use only.*